In [ ]:
import pandas as pd 

In [ ]:
wealth_25600 = pd.read_parquet("/share/ju/matt/mllmsci/outputs/pairwise/wealth_tester_large_20260304_183001.parquet")

In [ ]:
wealth_25600

In [ ]:
# drop rows with _r1 at end 
wealth_25600 = wealth_25600[~wealth_25600["pair_id"].str.endswith("_r1")]

In [ ]:
import subprocess
import sys
from collections import defaultdict

import numpy as np

# Install trueskill into the current notebook environment if needed.
import trueskill


recordings_md = pd.read_parquet(
    "/share/ju/matt/mllmsci/data/cyclomedia/recordings_manhattan_2025_md.parquet",
    columns=["imageId", "lat", "lon", "recordedAt", "orientation", "yawDegrees", "year"],
).rename(columns={"imageId": "recording_id"})

recordings_md.head()

In [ ]:
pair_df = wealth_25600.copy()

# Fallback if recording_id columns are absent.
if "recording_id_a" not in pair_df.columns:
    pair_df["recording_id_a"] = pair_df["sample_id_a"].astype(str).str.split("_").str[0]
if "recording_id_b" not in pair_df.columns:
    pair_df["recording_id_b"] = pair_df["sample_id_b"].astype(str).str.split("_").str[0]

# Keep required columns only.
matches = pair_df[["recording_id_a", "recording_id_b", "relative_score"]].dropna().copy()
matches["relative_score"] = pd.to_numeric(matches["relative_score"], errors="coerce").fillna(0).astype(int)

# Optional: deterministic order for reproducibility.
matches = matches.sort_values(["recording_id_a", "recording_id_b"]).reset_index(drop=True)

matches.head()

In [ ]:
env = trueskill.TrueSkill(draw_probability=0.05)
ratings = defaultdict(env.create_rating)

for row in matches.itertuples(index=False):
    a_id = str(row.recording_id_a)
    b_id = str(row.recording_id_b)
    score = int(row.relative_score)

    ra = ratings[a_id]
    rb = ratings[b_id]

    if score > 0:
        ra_new, rb_new = env.rate_1vs1(ra, rb, drawn=False)  # A wins
    elif score < 0:
        rb_new, ra_new = env.rate_1vs1(rb, ra, drawn=False)  # B wins
    else:
        ra_new, rb_new = env.rate_1vs1(ra, rb, drawn=True)   # draw

    ratings[a_id] = ra_new
    ratings[b_id] = rb_new

obs_counts = pd.concat(
    [matches["recording_id_a"], matches["recording_id_b"]],
    axis=0,
).value_counts()

image_ratings = pd.DataFrame(
    [
        {
            "recording_id": rid,
            "mu": r.mu,
            "sigma": r.sigma,
            "ts_point_estimate": r.mu,
            "ts_conservative": r.mu - 3.0 * r.sigma,
            "n_comparisons": int(obs_counts.get(rid, 0)),
        }
        for rid, r in ratings.items()
    ]
).sort_values("ts_conservative", ascending=False)

image_ratings.head(10)

In [ ]:
image_ratings_with_md = image_ratings.merge(
    recordings_md,
    on="recording_id",
    how="left",
)

# This is the per-image rating table ready for later tract/PUMA aggregation.
image_ratings_with_md.head(10)

In [ ]:
import geopandas as gpd

puma_gdf = gpd.read_file("/share/ju/matt/mllmsci/data/geo/nypuma2020_26a/nypuma2020.shp")[["PUMA", "geometry"]].copy()

recording_points = gpd.GeoDataFrame(
    recordings_md[["recording_id", "lat", "lon"]].dropna().copy(),
    geometry=gpd.points_from_xy(recordings_md["lon"], recordings_md["lat"]),
    crs="EPSG:4326",
).to_crs(puma_gdf.crs)

recordings_with_puma = gpd.sjoin(
    recording_points,
    puma_gdf,
    how="left",
    predicate="within",
).drop(columns=["index_right"], errors="ignore")

recordings_with_puma["PUMA"] = recordings_with_puma["PUMA"].astype("string")
recordings_with_puma.head()

In [ ]:
recording_to_puma = recordings_with_puma[["recording_id", "PUMA"]].drop_duplicates("recording_id")

puma_matches = (
    matches.merge(
        recording_to_puma.rename(columns={"recording_id": "recording_id_a", "PUMA": "puma_a"}),
        on="recording_id_a",
        how="left",
    )
    .merge(
        recording_to_puma.rename(columns={"recording_id": "recording_id_b", "PUMA": "puma_b"}),
        on="recording_id_b",
        how="left",
    )
)

# Keep only inter-PUMA comparisons for PUMA-vs-PUMA ratings.
puma_matches = puma_matches.dropna(subset=["puma_a", "puma_b"]).copy()
puma_matches = puma_matches[puma_matches["puma_a"] != puma_matches["puma_b"]].copy()

puma_pair_summary = (
    puma_matches.groupby(["puma_a", "puma_b"], as_index=False)
    .agg(
        n_comparisons=("relative_score", "size"),
        mean_relative_score=("relative_score", "mean"),
        median_relative_score=("relative_score", "median"),
    )
    .sort_values("n_comparisons", ascending=False)
)

puma_pair_summary.head(10)

In [ ]:
puma_env = trueskill.TrueSkill(draw_probability=0.05)
puma_ratings = defaultdict(puma_env.create_rating)

for row in puma_matches.itertuples(index=False):
    a = str(row.puma_a)
    b = str(row.puma_b)
    score = int(row.relative_score)

    ra = puma_ratings[a]
    rb = puma_ratings[b]

    if score > 0:
        ra_new, rb_new = puma_env.rate_1vs1(ra, rb, drawn=False)
    elif score < 0:
        rb_new, ra_new = puma_env.rate_1vs1(rb, ra, drawn=False)
    else:
        ra_new, rb_new = puma_env.rate_1vs1(ra, rb, drawn=True)

    puma_ratings[a] = ra_new
    puma_ratings[b] = rb_new

puma_obs_counts = pd.concat([puma_matches["puma_a"], puma_matches["puma_b"]], axis=0).value_counts()

puma_ratings_df = pd.DataFrame(
    [
        {
            "PUMA": puma,
            "mu": r.mu,
            "sigma": r.sigma,
            "ts_point_estimate": r.mu,
            "ts_conservative": r.mu - 3.0 * r.sigma,
            "n_comparisons": int(puma_obs_counts.get(puma, 0)),
        }
        for puma, r in puma_ratings.items()
    ]
).sort_values("ts_conservative", ascending=False)

puma_ratings_df.head(20)

In [ ]:
# drop rows in puma_ratings_df with n_comparisons < 30
N_COMPARISONS_THRESHOLD = 30 
puma_ratings_df = puma_ratings_df[puma_ratings_df["n_comparisons"] >= N_COMPARISONS_THRESHOLD]

In [ ]:
import re
import statsmodels.api as sm

acs_path = "/share/ju/matt/mllmsci/data/demo/ACSST5Y2024.S1901_2026-03-04T152819/ACSST5Y2024.S1901-Data.csv"
acs = pd.read_csv(acs_path, header=1)

# Keep NYC rows, then Manhattan-only rows (as requested).
acs_nyc = acs[acs["Geographic Area Name"].str.contains(r"NYC", case=False, na=False)].copy()
acs_nyc["PUMA"] = acs_nyc["Geography"].astype(str).str.extract(r"US\d{3}(\d{4})$")

acs_manhattan = acs_nyc[
    acs_nyc["Geographic Area Name"].str.contains(r"NYC-Manhattan", case=False, na=False)
].copy()

manhattan_pumas = set(acs_manhattan["PUMA"].dropna().astype(str).tolist())

# Income covariates.
income_cols = {
    "mean_income": "Estimate!!Households!!Mean income (dollars)",
    "median_income": "Estimate!!Households!!Median income (dollars)",
}

for k, c in income_cols.items():
    acs_manhattan[k] = pd.to_numeric(
        acs_manhattan[c].replace({"N": pd.NA, "-": pd.NA, "(X)": pd.NA}),
        errors="coerce",
    )

puma_covariates = acs_manhattan[["PUMA", "Geographic Area Name", *income_cols.keys()]].copy()
puma_covariates.head()

In [ ]:
# Rebuild pairwise competitions for Manhattan-only PUMAs.
puma_matches = puma_matches[
    puma_matches["puma_a"].astype(str).isin(manhattan_pumas)
    & puma_matches["puma_b"].astype(str).isin(manhattan_pumas)
].copy()

puma_pair_summary = (
    puma_matches.groupby(["puma_a", "puma_b"], as_index=False)
    .agg(
        n_comparisons=("relative_score", "size"),
        mean_relative_score=("relative_score", "mean"),
        median_relative_score=("relative_score", "median"),
    )
    .sort_values("n_comparisons", ascending=False)
)

# Recompute TrueSkill at Manhattan-PUMA level only.
puma_env = trueskill.TrueSkill(draw_probability=0.05)
puma_ratings = defaultdict(puma_env.create_rating)

for row in puma_matches.itertuples(index=False):
    a = str(row.puma_a)
    b = str(row.puma_b)
    score = int(row.relative_score)

    ra = puma_ratings[a]
    rb = puma_ratings[b]

    if score > 0:
        ra_new, rb_new = puma_env.rate_1vs1(ra, rb, drawn=False)
    elif score < 0:
        rb_new, ra_new = puma_env.rate_1vs1(rb, ra, drawn=False)
    else:
        ra_new, rb_new = puma_env.rate_1vs1(ra, rb, drawn=True)

    puma_ratings[a] = ra_new
    puma_ratings[b] = rb_new

puma_obs_counts = pd.concat([puma_matches["puma_a"], puma_matches["puma_b"]], axis=0).value_counts()

puma_ratings_df = pd.DataFrame(
    [
        {
            "PUMA": puma,
            "mu": r.mu,
            "sigma": r.sigma,
            "ts_point_estimate": r.mu,
            "ts_conservative": r.mu - 3.0 * r.sigma,
            "n_comparisons": int(puma_obs_counts.get(puma, 0)),
        }
        for puma, r in puma_ratings.items()
    ]
).sort_values("ts_conservative", ascending=False)

# Keep only Manhattan PUMAs for covariates + regression.
puma_ratings_df = puma_ratings_df[puma_ratings_df["PUMA"].astype(str).isin(manhattan_pumas)].copy()
puma_gdf = puma_gdf[puma_gdf["PUMA"].astype(str).isin(manhattan_pumas)].copy()

puma_model_df = puma_ratings_df.merge(puma_covariates, on="PUMA", how="inner")
puma_model_df = puma_model_df.dropna(subset=["ts_point_estimate", "mean_income", "median_income"]).copy()

y = puma_model_df["ts_point_estimate"].astype(float)
uni_models = {}
rows = []

for cov in ["mean_income", "median_income"]:
    X_cov = sm.add_constant(puma_model_df[[cov]].astype(float))
    fit = sm.OLS(y, X_cov).fit(cov_type="HC3")
    uni_models[cov] = {"fit": fit, "X": X_cov}
    rows.append(
        {
            "covariate": cov,
            "coef": fit.params[cov],
            "robust_se": fit.bse[cov],
            "p_value": fit.pvalues[cov],
            "r2": fit.rsquared,
            "adj_r2": fit.rsquared_adj,
            "n_obs": int(fit.nobs),
        }
    )

uni_summary = pd.DataFrame(rows).sort_values("p_value")
uni_summary

In [ ]:
# Pick one univariate fit for row-level diagnostics (median income by default).
diag_covariate = "median_income"
fit = uni_models[diag_covariate]["fit"]
X_diag = uni_models[diag_covariate]["X"]

puma_diag_df = puma_model_df.assign(
    ts_pred=fit.predict(X_diag),
    ts_resid=y - fit.predict(X_diag),
)

puma_diag_df[
    [
        "PUMA",
        "Geographic Area Name",
        "ts_point_estimate",
        "mean_income",
        "median_income",
        "ts_pred",
        "ts_resid",
        "n_comparisons",
    ]
].sort_values("ts_point_estimate", ascending=False).head(20)

In [ ]:
# Quick visualization for one univariate relationship.
import matplotlib.pyplot as plt

plot_df = puma_model_df.copy()

fig, ax = plt.subplots(1, 1, figsize=(7, 7))
ax.scatter(plot_df["median_income"], plot_df["ts_point_estimate"], alpha=0.8)
ax.set_title("PUMA TrueSkill vs Median Household Income")
ax.set_xlabel("Median household income (ACS S1901)")
ax.set_ylabel("TrueSkill point estimate")

# draw a perfect y=x line 
ax.plot(ax.get_xlim(), ax.get_ylim(), ls="--", c=".3")

plt.show()


In [ ]:
# Manhattan-only map comparison: median income vs TrueSkill.
import matplotlib.pyplot as plt

puma_ratings_gdf = puma_ratings_df.merge(puma_gdf, on="PUMA", how="left")
puma_covariates_gdf = puma_covariates.merge(puma_gdf, on="PUMA", how="left")

puma_ratings_gdf = gpd.GeoDataFrame(puma_ratings_gdf, geometry="geometry", crs=puma_gdf.crs)
puma_covariates_gdf = gpd.GeoDataFrame(puma_covariates_gdf, geometry="geometry", crs=puma_gdf.crs)

fig, ax = plt.subplots(2, 1, figsize=(10, 10))

puma_covariates_gdf.plot(column="median_income", cmap="viridis", ax=ax[0])
ax[0].set_title("Manhattan PUMAs: Median Household Income")
ax[0].set_axis_off()

puma_ratings_gdf.plot(column="ts_point_estimate", cmap="viridis", ax=ax[1])
ax[1].set_title("Manhattan PUMAs: TrueSkill Point Estimate")
ax[1].set_axis_off()

plt.show()



In [ ]:
# Census tract mapping (Manhattan-only scope).
tract_gdf = gpd.read_file(
    "/share/ju/matt/mllmsci/data/geo/2020_Census_Tracts_20260304.geojson"
)[["geoid", "boroname", "geometry"]].copy()

tract_gdf["geoid"] = tract_gdf["geoid"].astype(str)
tract_gdf["boroname"] = tract_gdf["boroname"].astype(str)
tract_gdf = tract_gdf.to_crs(recording_points.crs)

# Assign each recording to a tract.
recordings_with_tract = gpd.sjoin(
    recording_points,
    tract_gdf,
    how="left",
    predicate="within",
).drop(columns=["index_right"], errors="ignore")

recordings_with_tract["geoid"] = recordings_with_tract["geoid"].astype("string")

# Manhattan-only tract IDs from geometry source of truth.
manhattan_tracts = set(
    tract_gdf.loc[tract_gdf["boroname"].str.lower().eq("manhattan"), "geoid"]
    .dropna()
    .astype(str)
    .tolist()
)

recordings_with_tract.head()

In [ ]:
# Census tract income covariates with identical Manhattan-only filtering.
acs_ct_path = "/share/ju/matt/mllmsci/data/demo/ct/ACSST5Y2024.S1901-Data.csv"
acs_ct = pd.read_csv(acs_ct_path, header=1)

# NYC-ish filter first, then Manhattan-only (parallel to the PUMA section).
acs_ct_nyc = acs_ct[
    acs_ct["Geographic Area Name"].str.contains(r"County; New York$", case=False, na=False)
    | acs_ct["Geographic Area Name"].str.contains(r"New York County", case=False, na=False)
].copy()

acs_ct_nyc["geoid"] = acs_ct_nyc["Geography"].astype(str).str.extract(r"US(\d{11})$")

acs_ct_manhattan = acs_ct_nyc[
    acs_ct_nyc["Geographic Area Name"].str.contains(r"New York County", case=False, na=False)
].copy()
acs_ct_manhattan = acs_ct_manhattan[
    acs_ct_manhattan["geoid"].astype(str).isin(manhattan_tracts)
].copy()

income_cols_ct = {
    "mean_income": "Estimate!!Households!!Mean income (dollars)",
    "median_income": "Estimate!!Households!!Median income (dollars)",
}
for k, c in income_cols_ct.items():
    acs_ct_manhattan[k] = pd.to_numeric(
        acs_ct_manhattan[c].replace({"N": pd.NA, "-": pd.NA, "(X)": pd.NA}),
        errors="coerce",
    )

tract_covariates = acs_ct_manhattan[["geoid", "Geographic Area Name", *income_cols_ct.keys()]].copy()
tract_covariates.head()

In [ ]:
# Manhattan-only tract pair aggregation, TrueSkill, and univariate regressions.
recording_to_tract = recordings_with_tract[["recording_id", "geoid"]].drop_duplicates("recording_id")

tract_matches = (
    matches.merge(
        recording_to_tract.rename(columns={"recording_id": "recording_id_a", "geoid": "tract_a"}),
        on="recording_id_a",
        how="left",
    )
    .merge(
        recording_to_tract.rename(columns={"recording_id": "recording_id_b", "geoid": "tract_b"}),
        on="recording_id_b",
        how="left",
    )
)

tract_matches = tract_matches.dropna(subset=["tract_a", "tract_b"]).copy()
tract_matches = tract_matches[
    tract_matches["tract_a"].astype(str).isin(manhattan_tracts)
    & tract_matches["tract_b"].astype(str).isin(manhattan_tracts)
].copy()
tract_matches = tract_matches[tract_matches["tract_a"] != tract_matches["tract_b"]].copy()

tract_pair_summary = (
    tract_matches.groupby(["tract_a", "tract_b"], as_index=False)
    .agg(
        n_comparisons=("relative_score", "size"),
        mean_relative_score=("relative_score", "mean"),
        median_relative_score=("relative_score", "median"),
    )
    .sort_values("n_comparisons", ascending=False)
)

tract_env = trueskill.TrueSkill(draw_probability=0.05)
tract_ratings = defaultdict(tract_env.create_rating)

for row in tract_matches.itertuples(index=False):
    a = str(row.tract_a)
    b = str(row.tract_b)
    score = int(row.relative_score)

    ra = tract_ratings[a]
    rb = tract_ratings[b]

    if score > 0:
        ra_new, rb_new = tract_env.rate_1vs1(ra, rb, drawn=False)
    elif score < 0:
        rb_new, ra_new = tract_env.rate_1vs1(rb, ra, drawn=False)
    else:
        ra_new, rb_new = tract_env.rate_1vs1(ra, rb, drawn=True)

    tract_ratings[a] = ra_new
    tract_ratings[b] = rb_new

tract_obs_counts = pd.concat([tract_matches["tract_a"], tract_matches["tract_b"]], axis=0).value_counts()

tract_ratings_df = pd.DataFrame(
    [
        {
            "geoid": tract,
            "mu": r.mu,
            "sigma": r.sigma,
            "ts_point_estimate": r.mu,
            "ts_conservative": r.mu - 3.0 * r.sigma,
            "n_comparisons": int(tract_obs_counts.get(tract, 0)),
        }
        for tract, r in tract_ratings.items()
    ]
).sort_values("ts_conservative", ascending=False)

tract_ratings_df = tract_ratings_df[tract_ratings_df["geoid"].astype(str).isin(manhattan_tracts)].copy()
tract_model_df = tract_ratings_df.merge(tract_covariates, on="geoid", how="inner")
tract_model_df = tract_model_df.dropna(subset=["ts_point_estimate", "mean_income", "median_income"]).copy()

y_ct = tract_model_df["ts_point_estimate"].astype(float)
ct_uni_models = {}
ct_rows = []

for cov in ["mean_income", "median_income"]:
    X_cov = sm.add_constant(tract_model_df[[cov]].astype(float))
    fit = sm.OLS(y_ct, X_cov).fit(cov_type="HC3")
    ct_uni_models[cov] = {"fit": fit, "X": X_cov}
    ct_rows.append(
        {
            "covariate": cov,
            "coef": fit.params[cov],
            "robust_se": fit.bse[cov],
            "p_value": fit.pvalues[cov],
            "r2": fit.rsquared,
            "adj_r2": fit.rsquared_adj,
            "n_obs": int(fit.nobs),
        }
    )

tract_uni_summary = pd.DataFrame(ct_rows).sort_values("p_value")
tract_uni_summary

In [ ]:
# tufte-style map of the tracts colored by TrueSkill rating and median income 
tract_ratings_gdf = tract_ratings_df.merge(tract_gdf, on="geoid", how="left")
tract_ratings_gdf = gpd.GeoDataFrame(tract_ratings_gdf, geometry="geometry", crs=tract_gdf.crs)

tract_covariates_gdf = tract_covariates.merge(tract_gdf, on="geoid", how="left")
tract_covariates_gdf = gpd.GeoDataFrame(tract_covariates_gdf, geometry="geometry", crs=tract_gdf.crs)

fig, ax = plt.subplots(1, 2, figsize=(10, 10))

tract_ratings_gdf.plot(column="ts_point_estimate", cmap="viridis", ax=ax[0])
tract_covariates_gdf.plot(column="median_income", cmap="viridis", ax=ax[1])

ax[0].set_title("TrueSkill Rating")
ax[1].set_title("Median Income")
ax[0].set_axis_off()
ax[1].set_axis_off()
plt.show()



In [ ]:
# scatter plot of TrueSkill rating vs. median income with regression line 
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(10, 10))
ax.scatter(tract_model_df["median_income"], tract_model_df["ts_point_estimate"], alpha=0.8)
ax.set_title("TrueSkill Rating vs Median Household Income")
ax.set_xlabel("Median Household Income (ACS S1901)")

# add a regression line
x = tract_model_df["median_income"]
y = tract_model_df["ts_point_estimate"]
m, b = np.polyfit(x, y, 1)
ax.plot(x, m*x + b, color='red')

plt.show()
